# 03b - Topic modeling clásico: NMF y LDA

Esta notebook continúa el recorrido del Práctico 3. Después de probar KMeans con distintas representaciones, acá se exploran dos modelos clásicos de topic modeling: NMF y LDA.

La intención no es presentar estos modelos como solución final, sino entender qué aportan frente al clustering: en lugar de agrupar documentos en el espacio vectorial, buscan representar los textos como combinaciones de tópicos y describir cada tópico por sus palabras más importantes.

## Preparación del corpus

Se trabaja con `df_final.csv`, generado en el práctico anterior. Para mantener la reproducibilidad, los artefactos derivados se guardan en `data/processed/` y los resúmenes comparativos en `reports/`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords

from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

nltk.download("stopwords", quiet=True)

PROCESSED_DIR = Path("../data/processed") if Path.cwd().name == "notebooks" else Path("data/processed")
REPORTS_DIR = Path("../reports") if Path.cwd().name == "notebooks" else Path("reports")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [2]:
df = pd.read_csv(PROCESSED_DIR / "df_final.csv")
print(df.shape)
df[["rawContent_clean", "pysentimiento"]].head()

(8882, 78)


,rawContent_clean,pysentimiento
0,Boluda ahí dice que yo debería pesar 43kg eso ...,NEG
1,"Jugador de +30 años, vendehumo, con tendencias...",NEG
2,La tabla de mi pediatra cuando tenia 6 años y ...,NEG
3,chupame un huevo como voy a pesar 48 kg midien...,NEG
4,Eso me da curiosidad en la gente obesa. Miro k...,NEU


In [3]:
stop_words = set(stopwords.words("spanish"))
SEARCH_KEYWORDS = {
    "obesidad", "obesidaddd", "gordura", "sobrepeso", "gorduras",
    "adiposidad", "obesito", "obesoo", "gorrrdura", "obesaa",
    "obesita", "obesaaa", "obeso", "obesa", "obesos", "obesas",
}

custom_stopwords = SEARCH_KEYWORDS.union({
    "si", "no", "sos", "vos", "jaja", "ja", "q", "xq", "mas", "más", "solo", "la", "el",
    "ser", "tener", "hacer", "hace", "va", "re", "ahora", "bien", "mal", "puede", "pueden",
    "gente", "persona", "personas", "cosa", "cosas", "día", "dias", "días", "año", "años", "vez",
    "obesidad", "obeso", "obesa", "obesos", "obesas", "sobrepeso", "gordo", "gorda", "gordos", "gordas",
    "gordura", "mórbida", "morbida", "mórbido", "morbido",
    "vida", "siempre", "nunca", "dice", "decir", "dijo", "tan", "mejor", "nadie", "igual",
    "bueno", "buena", "tipo", "mismo", "misma", "voy", "anda", "mira", "entonces", "hoy",
    "veces", "ver", "vas", "después", "despues", "lado", "dan", "parece", "ganas", "tampoco",
    "menos", "ahí", "ahi", "hablar", "habla", "pasar", "falta", "creo", "cuenta", "dicen",
    "claro", "cada", "acá", "aca", "cómo", "como", "toda", "todo", "dos", "fin", "favor",
})
all_stopwords = stop_words.union(custom_stopwords)

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-záéíóúüñ0-9 ]+", " ", text)
    tokens = [token for token in text.split() if token not in all_stopwords and len(token) > 2]
    return " ".join(tokens)

df["topic_text"] = df["rawContent_clean"].apply(normalize_text)
docs = df["topic_text"].fillna("").tolist()

df[["rawContent_clean", "topic_text"]].head()


,rawContent_clean,topic_text
0,Boluda ahí dice que yo debería pesar 43kg eso ...,boluda debería pesar 43kg orto empezando cuida...
1,"Jugador de +30 años, vendehumo, con tendencias...",jugador vendehumo tendencias lesionarse tribun...
2,La tabla de mi pediatra cuando tenia 6 años y ...,tabla pediatra tenia decía
3,chupame un huevo como voy a pesar 48 kg midien...,chupame huevo pesar midiendo casi quién hizo l...
4,Eso me da curiosidad en la gente obesa. Miro k...,curiosidad miro kilos mortales veo placer toca...


## Representación documento-término

Para NMF se usa TF-IDF, porque el modelo trabaja bien con pesos no negativos que reducen el peso de términos demasiado frecuentes. Para LDA se usa una matriz de conteos, porque su formulación probabilística parte de frecuencias de palabras.

No se fuerzan embeddings Sentence-BERT en esta notebook: NMF y LDA son modelos de tópicos clásicos basados en términos. Los embeddings contextuales se retoman en la notebook de BERTopic.

In [4]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.9, min_df=10, ngram_range=(1, 2), max_features=5000)
count_vectorizer = CountVectorizer(max_df=0.9, min_df=10, ngram_range=(1, 2), max_features=5000)

X_tfidf = tfidf_vectorizer.fit_transform(docs)
X_count = count_vectorizer.fit_transform(docs)

print(f"Matriz TF-IDF para NMF: {X_tfidf.shape[0]} documentos x {X_tfidf.shape[1]} features")
print(f"Matriz de conteos para LDA: {X_count.shape[0]} documentos x {X_count.shape[1]} features")

Matriz TF-IDF para NMF: 8882 documentos x 1581 features
Matriz de conteos para LDA: 8882 documentos x 1581 features


## Funciones de evaluación

Se usan métricas simples pero comparables: diversidad de tópicos, NPMI aproximado sobre co-ocurrencias de términos, tamaño de tópicos y probabilidad media de asignación. En esta notebook se priorizan tablas sobre gráficos, porque las curvas no agregaban demasiado y hacían más pesada la lectura.


In [5]:
def top_terms_from_components(components, vectorizer, n_terms=12):
    feature_names = np.array(vectorizer.get_feature_names_out())
    rows = []
    for topic_id, component in enumerate(components):
        top_indices = component.argsort()[::-1][:n_terms]
        for rank, idx in enumerate(top_indices, start=1):
            rows.append({
                "topic": int(topic_id),
                "rank": rank,
                "term": feature_names[idx],
                "weight": float(component[idx]),
            })
    return pd.DataFrame(rows)

def topic_diversity(terms_df, top_n=10):
    top_terms = terms_df[terms_df["rank"] <= top_n]["term"].tolist()
    return len(set(top_terms)) / len(top_terms) if top_terms else np.nan

def mean_npmi(terms_df, binary_matrix, vectorizer, top_n=10):
    feature_index = {term: idx for idx, term in enumerate(vectorizer.get_feature_names_out())}
    binary_csc = binary_matrix.astype(bool).astype(int).tocsc()
    n_docs = binary_matrix.shape[0]
    scores = []
    for _, group in terms_df[terms_df["rank"] <= top_n].groupby("topic"):
        terms = [term for term in group.sort_values("rank")["term"] if term in feature_index]
        for i in range(len(terms)):
            for j in range(i + 1, len(terms)):
                idx_i = feature_index[terms[i]]
                idx_j = feature_index[terms[j]]
                col_i = binary_csc[:, idx_i]
                col_j = binary_csc[:, idx_j]
                p_i = col_i.sum() / n_docs
                p_j = col_j.sum() / n_docs
                p_ij = col_i.multiply(col_j).sum() / n_docs
                if p_ij > 0 and p_i > 0 and p_j > 0:
                    pmi = np.log(p_ij / (p_i * p_j))
                    scores.append(pmi / (-np.log(p_ij)))
    return float(np.mean(scores)) if scores else np.nan

def summarize_assignments(method, variant, doc_topic):
    topic = doc_topic.argmax(axis=1)
    probability = doc_topic.max(axis=1)
    sizes = pd.Series(topic).value_counts()
    return pd.DataFrame({
        "doc_id": df.index,
        "method": method,
        "variant": variant,
        "topic": topic,
        "probability": probability,
    }), {
        "method": method,
        "variant": variant,
        "n_topics": int(doc_topic.shape[1]),
        "min_topic_size": int(sizes.min()),
        "median_topic_size": float(sizes.median()),
        "max_topic_size": int(sizes.max()),
        "mean_assignment_probability": float(probability.mean()),
        "median_assignment_probability": float(np.median(probability)),
    }

def show_representative_docs(assignments_df, topic_id, n=5):
    selected = assignments_df[assignments_df["topic"] == topic_id].sort_values("probability", ascending=False).head(n)
    cols = ["rawContent_clean", "pysentimiento"]
    display(df.loc[selected["doc_id"], cols].assign(probability=selected["probability"].values))

## Selección de cantidad de tópicos

Se comparan varios valores de `k`. En NMF se mira el error de reconstrucción junto con diversidad/coherencia de términos. En LDA se mira perplexity/log-likelihood, también junto con diversidad/coherencia.

La selección no se toma como un óptimo automático. Para que la comparación con KMeans sea más directa, se conserva `k=12`: alcanza para mostrar una variedad razonable de tópicos sin volver ilegible la revisión cualitativa.


In [6]:
topic_range = [4, 6, 8, 10, 12, 15, 20, 25, 30]

nmf_sweep = []
for n_topics in topic_range:
    model = NMF(n_components=n_topics, init="nndsvda", random_state=RANDOM_STATE, max_iter=500)
    doc_topic = model.fit_transform(X_tfidf)
    terms_df = top_terms_from_components(model.components_, tfidf_vectorizer, n_terms=10)
    assignments_df, assignment_summary = summarize_assignments("NMF", f"nmf_tfidf_k{n_topics}", normalize(doc_topic, norm="l1"))
    nmf_sweep.append({
        **assignment_summary,
        "reconstruction_error": float(model.reconstruction_err_),
        "topic_diversity": topic_diversity(terms_df, top_n=10),
        "mean_npmi": mean_npmi(terms_df, X_count, count_vectorizer, top_n=10),
    })

nmf_sweep_df = pd.DataFrame(nmf_sweep)
nmf_sweep_df

,method,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,median_assignment_probability,reconstruction_error,topic_diversity,mean_npmi
0,NMF,nmf_tfidf_k4,4,382,658.5,7183,0.704176,0.748729,89.504867,0.975000,0.168198
1,NMF,nmf_tfidf_k6,6,373,626.0,5808,0.580086,0.552791,89.118972,0.900000,0.167717
2,NMF,nmf_tfidf_k8,8,316,825.5,3391,0.495021,0.438702,88.740641,0.850000,0.156720
3,NMF,nmf_tfidf_k10,10,297,600.0,3099,0.474504,0.405325,88.406297,0.790000,0.153501
4,NMF,nmf_tfidf_k12,12,156,297.0,4706,0.572770,0.566137,88.085039,0.816667,0.201577
5,NMF,nmf_tfidf_k15,15,153,283.0,3265,0.532705,0.504309,87.640392,0.813333,0.196850
6,NMF,nmf_tfidf_k20,20,130,313.0,1358,0.464204,0.409566,86.959074,0.780000,0.212181
7,NMF,nmf_tfidf_k25,25,120,282.0,793,0.448544,0.381695,86.333319,0.788000,0.231181
8,NMF,nmf_tfidf_k30,30,100,229.5,788,0.440586,0.367994,85.722131,0.770000,0.235636


In [7]:
nmf_sweep_df[[
    "variant",
    "n_topics",
    "min_topic_size",
    "median_topic_size",
    "max_topic_size",
    "mean_assignment_probability",
    "reconstruction_error",
    "topic_diversity",
    "mean_npmi",
]].sort_values("mean_npmi", ascending=False)


,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,reconstruction_error,topic_diversity,mean_npmi
8,nmf_tfidf_k30,30,100,229.5,788,0.440586,85.722131,0.770000,0.235636
7,nmf_tfidf_k25,25,120,282.0,793,0.448544,86.333319,0.788000,0.231181
6,nmf_tfidf_k20,20,130,313.0,1358,0.464204,86.959074,0.780000,0.212181
4,nmf_tfidf_k12,12,156,297.0,4706,0.572770,88.085039,0.816667,0.201577
5,nmf_tfidf_k15,15,153,283.0,3265,0.532705,87.640392,0.813333,0.196850
0,nmf_tfidf_k4,4,382,658.5,7183,0.704176,89.504867,0.975000,0.168198
1,nmf_tfidf_k6,6,373,626.0,5808,0.580086,89.118972,0.900000,0.167717
2,nmf_tfidf_k8,8,316,825.5,3391,0.495021,88.740641,0.850000,0.156720
3,nmf_tfidf_k10,10,297,600.0,3099,0.474504,88.406297,0.790000,0.153501


In [8]:
lda_sweep = []
for n_topics in topic_range:
    model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=RANDOM_STATE,
        learning_method="batch",
        max_iter=20,
        n_jobs=-1,
    )
    doc_topic = model.fit_transform(X_count)
    terms_df = top_terms_from_components(model.components_, count_vectorizer, n_terms=10)
    assignments_df, assignment_summary = summarize_assignments("LDA", f"lda_count_k{n_topics}", doc_topic)
    lda_sweep.append({
        **assignment_summary,
        "perplexity": float(model.perplexity(X_count)),
        "log_likelihood": float(model.score(X_count)),
        "topic_diversity": topic_diversity(terms_df, top_n=10),
        "mean_npmi": mean_npmi(terms_df, X_count, count_vectorizer, top_n=10),
    })

lda_sweep_df = pd.DataFrame(lda_sweep)
lda_sweep_df

,method,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,median_assignment_probability,perplexity,log_likelihood,topic_diversity,mean_npmi
0,LDA,lda_count_k4,4,1855,2222.5,2582,0.627882,0.624834,1442.345121,-310317.207153,0.850000,0.082115
1,LDA,lda_count_k6,6,1171,1350.5,2087,0.594287,0.583111,1536.985367,-313028.424570,0.883333,0.081288
2,LDA,lda_count_k8,8,866,1067.0,1723,0.565522,0.562424,1610.233924,-315014.572210,0.887500,0.084609
3,LDA,lda_count_k10,10,628,856.5,1470,0.551070,0.549986,1669.572018,-316558.382259,0.870000,0.097053
4,LDA,lda_count_k12,12,510,724.0,1364,0.532101,0.541663,1695.316927,-317211.197264,0.883333,0.109109
5,LDA,lda_count_k15,15,397,555.0,1219,0.505090,0.516667,1734.869381,-318195.063376,0.946667,0.143363
6,LDA,lda_count_k20,20,351,417.5,1076,0.471742,0.499738,1860.019115,-321166.599926,0.945000,0.160020
7,LDA,lda_count_k25,25,245,317.0,1111,0.447430,0.449362,1933.151884,-322811.821679,0.956000,0.171676
8,LDA,lda_count_k30,30,209,261.0,964,0.425317,0.406667,2045.627930,-325224.430368,0.960000,0.181646


In [9]:
lda_sweep_df[[
    "variant",
    "n_topics",
    "min_topic_size",
    "median_topic_size",
    "max_topic_size",
    "mean_assignment_probability",
    "perplexity",
    "topic_diversity",
    "mean_npmi",
]].sort_values("mean_npmi", ascending=False)


,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,perplexity,topic_diversity,mean_npmi
8,lda_count_k30,30,209,261.0,964,0.425317,2045.627930,0.960000,0.181646
7,lda_count_k25,25,245,317.0,1111,0.447430,1933.151884,0.956000,0.171676
6,lda_count_k20,20,351,417.5,1076,0.471742,1860.019115,0.945000,0.160020
5,lda_count_k15,15,397,555.0,1219,0.505090,1734.869381,0.946667,0.143363
4,lda_count_k12,12,510,724.0,1364,0.532101,1695.316927,0.883333,0.109109
3,lda_count_k10,10,628,856.5,1470,0.551070,1669.572018,0.870000,0.097053
2,lda_count_k8,8,866,1067.0,1723,0.565522,1610.233924,0.887500,0.084609
0,lda_count_k4,4,1855,2222.5,2582,0.627882,1442.345121,0.850000,0.082115
1,lda_count_k6,6,1171,1350.5,2087,0.594287,1536.985367,0.883333,0.081288


Las métricas permiten comparar, pero no resuelven solas el problema. En topic modeling interesa que los tópicos sean legibles, que no repitan siempre las mismas palabras y que tengan una distribución razonable de documentos.

En las tablas se ve que aumentar la cantidad de tópicos puede mejorar alguna métrica puntual, pero también vuelve más fragmentada la lectura. Por eso se mantiene `k=12` como compromiso de comparabilidad e interpretabilidad, no como una verdad matemática del corpus.


In [10]:
NMF_TOPICS = 12
LDA_TOPICS = 12

nmf_model = NMF(n_components=NMF_TOPICS, init="nndsvda", random_state=RANDOM_STATE, max_iter=500)
nmf_doc_topic_raw = nmf_model.fit_transform(X_tfidf)
nmf_doc_topic = normalize(nmf_doc_topic_raw, norm="l1")
nmf_terms = top_terms_from_components(nmf_model.components_, tfidf_vectorizer, n_terms=12)
nmf_assignments, nmf_summary = summarize_assignments("NMF", f"nmf_tfidf_k{NMF_TOPICS}", nmf_doc_topic)
nmf_summary.update({
    "reconstruction_error": float(nmf_model.reconstruction_err_),
    "perplexity": np.nan,
    "log_likelihood": np.nan,
    "topic_diversity": topic_diversity(nmf_terms, top_n=10),
    "mean_npmi": mean_npmi(nmf_terms, X_count, count_vectorizer, top_n=10),
})

display(nmf_terms.pivot(index="topic", columns="rank", values="term"))

rank,1,2,3,4,5,6,7,8,9,10,11,12
topic,,,,,,,,,,,,
0,comer,feliz,debe,quiero,siento,dejar,semana,dejar comer,dulce,comer dulce,dormir,darle
1,mierda,puta,puto,asco,comen,hdp,cabeza,negro,pelotudo,panza,pelotas,pija
2,enfermedad,diabetes,mental,grave,crónica,además,riesgo,alguien,basta,elige,sino,anorexia
3,peso,bajar,bajar peso,cuestión,cuestión peso,mido,ideal,peso ideal,saludable,normal,bajo,nutricionista
4,dios,siento,mio,dios mio,puta,asi,asco,quiero,nivel,paso,momento,puedo
5,así,niños,etc,total,todas,asco,mil,feliz,necesito,pedazo,desagradable,aún
6,tenes,cara,podes,puta,adentro,foto,vieja,deja,nota,arterias,neuronas,vergüenza
7,problemas,salud,problemas salud,diabetes,enfermedades,riesgo,etc,mundial,trae,mental,gordofobia,usa
8,hambre,van,mucha,pueblo,todas,país,pobreza,congreso,pasan,niños,colesterol,masa


In [11]:
lda_model = LatentDirichletAllocation(
    n_components=LDA_TOPICS,
    random_state=RANDOM_STATE,
    learning_method="batch",
    max_iter=20,
    n_jobs=-1,
)
lda_doc_topic = lda_model.fit_transform(X_count)
lda_terms = top_terms_from_components(lda_model.components_, count_vectorizer, n_terms=12)
lda_assignments, lda_summary = summarize_assignments("LDA", f"lda_count_k{LDA_TOPICS}", lda_doc_topic)
lda_summary.update({
    "reconstruction_error": np.nan,
    "perplexity": float(lda_model.perplexity(X_count)),
    "log_likelihood": float(lda_model.score(X_count)),
    "topic_diversity": topic_diversity(lda_terms, top_n=10),
    "mean_npmi": mean_npmi(lda_terms, X_count, count_vectorizer, top_n=10),
})

display(lda_terms.pivot(index="topic", columns="rank", values="term"))

rank,1,2,3,4,5,6,7,8,9,10,11,12
topic,,,,,,,,,,,,
0,enfermedad,digo,salud,asco,mundial,tetas,gordofobia,razón,hilo,pesar,paso,dije
1,todas,problema,quiero,comer,ustedes,tema,flaco,leche,además,sino,idea,deja
2,gato,mujer,hizo,vamos,gordito,culo,terrible,gobierno,hombre,respeto,cara,basta
3,peso,kilos,bajar,normal,bajar peso,baja,cuerpo,furia,altura,así,masa,mientras
4,tratamiento,veo,siento,saludable,100,diabetes,debe,casi,seguro,ley,alimentación,peor
5,peso,riesgo,física,cuestión,chica,alguna,negro,cáncer,sueño,argentina,factores,actividad
6,problemas,salud,diabetes,dios,enfermedad,alguien,van,podes,mundo,alimentación,mayoría,mala
7,salud,puedo,tiempo,quiere,enfermedades,cirugía,hinchazón,problema,trabajo,crer,centro,aparte
8,mierda,así,asi,mujeres,encima,importa,país,comer,che,hombres,etc,niños


## Ejemplos representativos

Además de mirar palabras, se revisan tweets con alta probabilidad de pertenecer a algunos tópicos. Esto ayuda a detectar si las palabras representativas realmente se traducen en documentos coherentes.

In [12]:
for topic_id in [0, 1, 2]:
    print(f"NMF - tópico {topic_id}")
    show_representative_docs(nmf_assignments, topic_id, n=4)

NMF - tópico 0


,rawContent_clean,pysentimiento,probability
8696,El chelo no para de comer.esta rosando la obes...,NEG,1.0
8753,"Mi papá siempre decía: ""si no fuera por la gor...",NEU,1.0
6834,la del horóscopo me dijo básicamente q deje de...,NEG,1.0
7821,"soñe q era obesa, la vida me manda un lindo in...",NEU,1.0


NMF - tópico 1


,rawContent_clean,pysentimiento,probability
8576,"Borja obeso de mierda volvete a Narcolombia, h...",NEG,1.0
971,y si son incogible obesa de mierda,NEG,1.0
932,Dicen que le tiró mierda y apoya al obeso tran...,NEG,1.0
4126,q mierda decís obeso,NEG,1.0


NMF - tópico 2


,rawContent_clean,pysentimiento,probability
1941,Tener sobrepeso es una enfermedad y no una ofensa,NEG,1.0
2016,"La obesidad , es una enfermedad. Muy lejos de ...",NEG,1.0
1676,Porque la obesidad es un a enfermedad y hay qu...,NEG,1.0
1311,La obesidad es una enfermedad,NEG,1.0


In [13]:
for topic_id in [0, 1, 2]:
    print(f"LDA - tópico {topic_id}")
    show_representative_docs(lda_assignments, topic_id, n=4)

LDA - tópico 0


,rawContent_clean,pysentimiento,probability
7865,"La semaglutida, como dijimos, fue aprobada par...",NEU,0.932272
622,Una mina se sube al bondi con 5 crias y solo t...,NEG,0.929484
8370,"Es sencilla, no mezcles la salud y/o la estéti...",NEG,0.923609
2562,En 48 casos fallecidos se registraron comorbil...,NEG,0.916666


LDA - tópico 1


,rawContent_clean,pysentimiento,probability
2501,"FURIA: Catalina quiere estar en la cama, con e...",NEU,0.942707
2531,"Chicos se refiere al grupo de ella JAKSJAK, es...",NEG,0.942707
2404,CUÁNTO es un sueldo promedio por 6hs lun-vier ...,NEU,0.938886
1451,"Mi pueblo es chico, pero no me habló con las c...",NEU,0.934521


LDA - tópico 2


,rawContent_clean,pysentimiento,probability
1717,El Día Mundial de la busca concientizar a las ...,NEG,0.946076
2290,En serio comparas a Thor un hombre real o sea ...,NEG,0.934523
2865,"O sea, se queja que no es atractiva para los h...",NEG,0.934523
7607,Una chica de color toma la iniciativa y maneja...,NEU,0.934522


## Salidas para comparación

Se guardan tres salidas: métricas de barrido y selección, términos por tópico y asignaciones documento-tópico. Así la página final puede comparar KMeans, NMF, LDA y BERTopic con el mismo tipo de insumos.

In [14]:
classic_sweep_df = pd.concat([nmf_sweep_df, lda_sweep_df], ignore_index=True, sort=False)
classic_metrics_df = pd.DataFrame([nmf_summary, lda_summary])
classic_terms_df = pd.concat([
    nmf_terms.assign(method="NMF", variant=f"nmf_tfidf_k{NMF_TOPICS}"),
    lda_terms.assign(method="LDA", variant=f"lda_count_k{LDA_TOPICS}"),
], ignore_index=True)
classic_assignments_df = pd.concat([nmf_assignments, lda_assignments], ignore_index=True)

classic_sweep_df.to_csv(REPORTS_DIR / "classic_topic_model_sweep.csv", index=False)
classic_metrics_df.to_csv(REPORTS_DIR / "classic_topic_model_metrics.csv", index=False)
classic_terms_df.to_csv(REPORTS_DIR / "classic_topic_terms.csv", index=False)
classic_assignments_df.to_csv(PROCESSED_DIR / "classic_topic_document_assignments.csv", index=False)

display(classic_metrics_df)
display(classic_terms_df.head(20))
print("Guardado:")
print(REPORTS_DIR / "classic_topic_model_sweep.csv")
print(REPORTS_DIR / "classic_topic_model_metrics.csv")
print(REPORTS_DIR / "classic_topic_terms.csv")
print(PROCESSED_DIR / "classic_topic_document_assignments.csv")

,method,variant,n_topics,min_topic_size,median_topic_size,max_topic_size,mean_assignment_probability,median_assignment_probability,reconstruction_error,perplexity,log_likelihood,topic_diversity,mean_npmi
0,NMF,nmf_tfidf_k12,12,156,297.0,4706,0.572770,0.566137,88.085039,NaN,NaN,0.816667,0.201577
1,LDA,lda_count_k12,12,510,724.0,1364,0.532101,0.541663,NaN,1695.316927,-317211.197264,0.883333,0.109109


,topic,rank,term,weight,method,variant
0,0,1,comer,5.263409,NMF,nmf_tfidf_k12
1,0,2,feliz,0.488190,NMF,nmf_tfidf_k12
2,0,3,debe,0.486492,NMF,nmf_tfidf_k12
3,0,4,quiero,0.440266,NMF,nmf_tfidf_k12
4,0,5,siento,0.319999,NMF,nmf_tfidf_k12
5,0,6,dejar,0.230021,NMF,nmf_tfidf_k12
6,0,7,semana,0.215560,NMF,nmf_tfidf_k12
7,0,8,dejar comer,0.211614,NMF,nmf_tfidf_k12
8,0,9,dulce,0.203112,NMF,nmf_tfidf_k12
9,0,10,comer dulce,0.194093,NMF,nmf_tfidf_k12


Guardado:
../reports/classic_topic_model_sweep.csv
../reports/classic_topic_model_metrics.csv
../reports/classic_topic_terms.csv
../data/processed/classic_topic_document_assignments.csv


## Cierre

NMF y LDA aportan una capa interpretativa que KMeans no trae de forma directa: cada tópico queda definido por palabras y cada documento puede entenderse como mezcla de tópicos. En esta versión se dejaron solo tablas y ejemplos, porque los gráficos de términos no agregaban información suficiente.

Aun con stopwords ampliadas, aparecen tópicos amplios y algunas repeticiones, algo esperable en un corpus de tweets corto y ruidoso. Esta limitación justifica pasar a BERTopic, que combina embeddings contextuales con clustering y representación por términos.
